# Portfolio Optimization & Risk Analysis — Step-by-Step Tutorial

> **Author:** Portfolio Optimization System  
> **Objective:** Understand the mathematics behind Markowitz Mean-Variance Optimization, the Efficient Frontier, Monte Carlo simulation, and risk measures — with real equity data.

---

## Table of Contents
1. [Setup & Data](#1-setup)
2. [Log-Returns & Why We Use Them](#2-returns)
3. [Expected Returns & Covariance Matrix](#3-stats)
4. [Portfolio Mathematics](#4-portfolio-math)
5. [Markowitz Optimization — From Scratch](#5-markowitz)
6. [The Efficient Frontier](#6-frontier)
7. [Capital Market Line & Maximum Sharpe Portfolio](#7-cml)
8. [Monte Carlo Portfolio Simulation](#8-mc)
9. [Forward Path Simulation (GBM)](#9-gbm)
10. [Risk Measures: VaR & CVaR](#10-risk)
11. [Final Dashboard](#11-dashboard)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from data_fetcher        import fetch_price_data, compute_log_returns, annualise_returns, annualise_covariance, summary_statistics
from portfolio_optimizer import MeanVarianceOptimizer
from monte_carlo         import simulate_random_portfolios, simulate_gbm_paths
from visualizer          import (plot_efficient_frontier, plot_risk_return_scatter,
                                  plot_portfolio_weights, plot_gbm_simulation,
                                  plot_correlation_heatmap, plot_rolling_sharpe)

print('All modules loaded successfully!')

---
## 1. Setup & Data <a id='1-setup'></a>

We select 8 large-cap US equities spanning Technology, Consumer, Financials, Healthcare, and Energy sectors.  
Sector diversification is intentional — assets in different sectors tend to have lower pairwise correlations.

In [ ]:
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'JPM', 'JNJ', 'XOM']
RF      = 0.04   # 4% annualised risk-free rate (approximate US T-bill rate)

# Fetch from Yahoo Finance (cached to data/ folder)
prices = fetch_price_data(TICKERS, start='2019-01-01', end='2024-12-31', cache_dir='../data/')

print(f'Price data shape: {prices.shape}  ({prices.shape[0]} trading days)')
prices.tail()

In [ ]:
# Visualise normalised prices (base = 100 at start)
fig, ax = plt.subplots(figsize=(13, 5))
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')

normalised = (prices / prices.iloc[0]) * 100
colors = plt.cm.tab10(np.linspace(0, 1, len(TICKERS)))

for ticker, color in zip(TICKERS, colors):
    ax.plot(normalised.index, normalised[ticker], label=ticker, linewidth=1.5, color=color)

ax.axhline(100, color='white', linewidth=0.8, linestyle='--', alpha=0.4)
ax.set_title('Normalised Price History (Base = 100)', color='white', fontsize=13, fontweight='bold')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#21262d')
ax.grid(True, color='#21262d', alpha=0.5)
ax.legend(loc='upper left', fontsize=9, framealpha=0.3, labelcolor='white')
plt.tight_layout()
plt.show()

---
## 2. Log-Returns & Why We Use Them <a id='2-returns'></a>

### Simple vs Log Returns

**Simple return:**  
$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

**Log return:**  
$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln(P_t) - \ln(P_{t-1})$$

### Why log returns?

| Property | Simple | Log |
|----------|--------|-----|
| Time-additive | ❌ | ✅ $r_{1,T} = \sum_t r_t$ |
| Symmetric | ❌ | ✅ |
| Approx. normal | ≈ | ✅ |
| Prevents $P<0$ in simulation | ❌ | ✅ |

**Note:** For small returns, $r_t \approx R_t$, so the distinction matters less for daily data.

In [ ]:
log_ret  = compute_log_returns(prices)
simp_ret = prices.pct_change().dropna()

print(f'Log-returns shape: {log_ret.shape}')
print('\nFirst 3 rows of log-returns:')
log_ret.head(3).round(5)

In [ ]:
# Visualise return distribution — testing normality assumption
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')

colors = plt.cm.tab10(np.linspace(0, 1, len(TICKERS)))

for ax, ticker, color in zip(axes.flat, TICKERS, colors):
    ax.set_facecolor('#161b22')
    ax.hist(log_ret[ticker] * 100, bins=60, color=color, alpha=0.7, edgecolor='none')
    ax.set_title(ticker, color='white', fontsize=10)
    ax.tick_params(colors='white', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#21262d')

fig.suptitle('Daily Log-Return Distributions', color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Expected Returns & Covariance Matrix <a id='3-stats'></a>

### Expected Returns (annualised)

The **sample mean** of daily log-returns, scaled to annual:

$$\hat{\mu}_i = \frac{1}{T}\sum_{t=1}^{T} r_{i,t} \times 252$$

### Sample Covariance Matrix (annualised)

$$\hat{\Sigma}_{ij} = \frac{1}{T-1}\sum_{t=1}^{T}(r_{i,t}-\hat{\mu}_i)(r_{j,t}-\hat{\mu}_j) \times 252$$

- **Diagonal entries** $\Sigma_{ii} = \sigma_i^2$ are the asset variances
- **Off-diagonal entries** $\Sigma_{ij}$ are the covariances — they drive diversification
- **Correlation** $\rho_{ij} = \Sigma_{ij}/(\sigma_i \sigma_j)$ normalises to $[-1, +1]$

In [ ]:
mu    = annualise_returns(log_ret)
Sigma = annualise_covariance(log_ret)

print('Annualised Expected Returns:')
print((mu * 100).round(2).to_string())

print('\nAnnualised Covariance Matrix (%):')
(Sigma * 100).round(3)

In [ ]:
print('Summary Statistics:')
summary_statistics(prices)

In [ ]:
# Correlation heatmap
fig = plot_correlation_heatmap(log_ret)
plt.show()

---
## 4. Portfolio Mathematics <a id='4-portfolio-math'></a>

A **portfolio** is defined by a weight vector $\mathbf{w} = (w_1, \ldots, w_N)^\top$ where:
$$\sum_{i=1}^{N} w_i = 1, \quad w_i \geq 0 \text{ (long-only)}$$

### Portfolio Expected Return
$$\mu_p = \mathbf{w}^\top \boldsymbol{\mu}$$
A simple weighted average of asset returns.

### Portfolio Variance
$$\sigma_p^2 = \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} = \sum_i w_i^2 \sigma_i^2 + 2\sum_{i<j} w_i w_j \sigma_{ij}$$

The cross terms $\sigma_{ij}$ are the key to **diversification**: when assets are negatively correlated, the portfolio variance can be *lower than any individual asset's variance*.

### Sharpe Ratio
$$SR = \frac{\mu_p - r_f}{\sigma_p}$$
Measures **excess return per unit of total risk**. It is the slope of the Capital Market Line.

In [ ]:
# Demonstrate portfolio math from scratch
w_eq = np.ones(len(TICKERS)) / len(TICKERS)   # equal weight

mu_p    = w_eq @ mu.values                          # scalar dot product
sigma2_p = w_eq @ Sigma.values @ w_eq               # quadratic form
sigma_p  = np.sqrt(sigma2_p)
sharpe   = (mu_p - RF) / sigma_p

print('=== Equal-Weight Portfolio ===')
print(f'Expected Return (ann.)  : {mu_p*100:.2f}%')
print(f'Variance (ann.)         : {sigma2_p:.6f}')
print(f'Volatility (ann.)       : {sigma_p*100:.2f}%')
print(f'Sharpe Ratio            : {sharpe:.4f}')

# Show diversification benefit
vol_indiv = np.sqrt(np.diag(Sigma.values))
avg_vol   = (w_eq @ vol_indiv)
print(f'\nWeighted-avg. individual vol: {avg_vol*100:.2f}%')
print(f'Portfolio vol            : {sigma_p*100:.2f}%')
print(f'Diversification benefit  : {(avg_vol - sigma_p)*100:.2f}% reduction in vol')

---
## 5. Markowitz Optimization — From Scratch <a id='5-markowitz'></a>

### Quadratic Programming Formulation

For a target return $\mu^*$, the minimum-variance portfolio solves:

$$\min_{\mathbf{w}} \quad \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}$$

$$\text{subject to} \quad \mathbf{w}^\top \boldsymbol{\mu} = \mu^*, \quad \mathbf{1}^\top \mathbf{w} = 1, \quad \mathbf{w} \geq 0$$

### Analytical Lagrangian (without non-negativity constraint)

Form the Lagrangian:
$$\mathcal{L}(\mathbf{w}, \lambda_1, \lambda_2) = \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} + \lambda_1(\mu^* - \mathbf{w}^\top \boldsymbol{\mu}) + \lambda_2(1 - \mathbf{1}^\top \mathbf{w})$$

First-order conditions $\partial\mathcal{L}/\partial\mathbf{w} = 0$:
$$2\boldsymbol{\Sigma}\mathbf{w} = \lambda_1 \boldsymbol{\mu} + \lambda_2 \mathbf{1}$$
$$\mathbf{w}^* = \frac{1}{2}\boldsymbol{\Sigma}^{-1}(\lambda_1 \boldsymbol{\mu} + \lambda_2 \mathbf{1})$$

With long-only constraints ($w_i \geq 0$), we use **SLSQP** numerical optimisation.

In [ ]:
# Create the optimizer
opt = MeanVarianceOptimizer(mu, Sigma, rf=RF, allow_short=False)

# Global Minimum Variance Portfolio
gmv = opt.minimum_variance_portfolio()
print(gmv.summary())

In [ ]:
# Maximum Sharpe Ratio Portfolio (Tangency Portfolio)
msr = opt.maximum_sharpe_portfolio()
print(msr.summary())

In [ ]:
# Equal-Weight Benchmark
ew = opt.equal_weight_portfolio()
print(ew.summary())

---
## 6. The Efficient Frontier <a id='6-frontier'></a>

The **efficient frontier** is the set of portfolios that minimise variance for each level of target return. Tracing it:

1. Solve the GMV problem → get minimum feasible return $\mu^*_{\min}$
2. For each $\mu^* \in [\mu^*_{\min}, \mu^*_{\max}]$, solve the constrained QP
3. Collect $(\sigma_p, \mu_p)$ pairs → this traces the frontier

**Portfolios below the GMV are sub-optimal** — they accept lower return for the same risk.

In [ ]:
ef = opt.efficient_frontier(n_points=300)
print(f'Efficient frontier: {len(ef)} portfolios computed')
ef[['ret', 'vol', 'sharpe']].describe().round(4)

In [ ]:
# Plot just the frontier (no MC yet)
fig = plot_efficient_frontier(ef_df=ef, gmv=gmv, msr=msr, ew=ew, rf=RF)
plt.show()

---
## 7. Capital Market Line & Maximum Sharpe Portfolio <a id='7-cml'></a>

When a **risk-free asset** exists (e.g. US T-bills at rate $r_f$), investors can combine it with any risky portfolio $P$ to get a straight line in $(\sigma, \mu)$ space:

$$\mu_{\text{CML}}(\sigma) = r_f + \frac{\mu_P - r_f}{\sigma_P} \cdot \sigma$$

The optimal combination is the **tangency point** where this line is tangent to the efficient frontier. That point is the **Maximum Sharpe Ratio (MSR) portfolio**, because the slope of the CML equals the Sharpe Ratio:

$$\text{slope} = \frac{\mu_P - r_f}{\sigma_P} = SR_P$$

**Analytical solution** (no short-selling constraints):  
$$\mathbf{y}^* = \boldsymbol{\Sigma}^{-1}(\boldsymbol{\mu} - r_f \mathbf{1})$$
$$\mathbf{w}^*_{\text{MSR}} = \frac{\mathbf{y}^*}{\mathbf{1}^\top \mathbf{y}^*}$$

In [ ]:
# Verify the analytical intuition manually
excess = mu.values - RF
Sigma_inv = np.linalg.inv(Sigma.values)
y_star = Sigma_inv @ excess

# For long-only portfolios, negative weights get clipped by the optimizer
print('y* (unnormalised, analytical):')
for t, y in zip(TICKERS, y_star):
    print(f'  {t}: {y:.4f}')

# Our optimizer handles the inequality constraints numerically
print(f'\nMax Sharpe portfolio Sharpe: {msr.sharpe:.4f}')
print(f'Equal-weight Sharpe:         {ew.sharpe:.4f}')
print(f'GMV Sharpe:                  {gmv.sharpe:.4f}')

---
## 8. Monte Carlo Portfolio Simulation <a id='8-mc'></a>

Instead of solving the QP analytically, we can also **randomly sample** thousands of weight vectors from the simplex:

$$\mathbf{w} \sim \text{Dirichlet}(\mathbf{1}_N)$$

This is equivalent to drawing $N$ uniform samples and normalising:
$$u_i \sim U[0,1], \quad w_i = u_i / \sum_j u_j$$

We compute $(\sigma_p, \mu_p, SR_p)$ for each random portfolio. The resulting cloud **illustrates the feasible set** and makes the efficient frontier visually apparent as the upper-left envelope.

In [ ]:
mc = simulate_random_portfolios(mu, Sigma, n_portfolios=25_000, rf=RF, seed=42)

mc_df = mc.to_dataframe()
print(f'MC simulation: {len(mc_df)} portfolios')
mc_df[['return', 'vol', 'sharpe']].describe().round(4)

In [ ]:
# Full efficient frontier chart with MC cloud
fig = plot_efficient_frontier(ef_df=ef, gmv=gmv, msr=msr, ew=ew,
                               mc_result=mc, rf=RF)
plt.show()

In [ ]:
# Risk vs Return scatter
fig = plot_risk_return_scatter(mc_result=mc, gmv=gmv, msr=msr)
plt.show()

In [ ]:
# Compare portfolio weights
fig = plot_portfolio_weights({'Max Sharpe': msr, 'Min Variance': gmv, 'Equal Weight': ew})
plt.show()

---
## 9. Forward Path Simulation — Geometric Brownian Motion <a id='9-gbm'></a>

To forecast portfolio performance, we simulate future price paths under **Multivariate GBM**.

Each asset price follows:
$$dS_i = \mu_i S_i \, dt + \sigma_i S_i \, dW_i$$

In discrete log-return form (exact solution for GBM):
$$r_i(t) = \underbrace{\left(\mu_i - \frac{\sigma_i^2}{2}\right)}_\text{Itô drift} dt + \sigma_i \sqrt{dt} \, Z_i$$

The $-\sigma_i^2/2$ **Itô correction** is critical: without it, the geometric mean of $e^r$ would be biased.

### Generating Correlated Shocks

To inject the empirical correlation structure:
$$\mathbf{Z}_{\text{corr}} = \mathbf{L} \mathbf{Z}_{\text{iid}}, \quad \mathbf{Z}_{\text{iid}} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$
$$\text{where} \quad \boldsymbol{\Sigma} = \mathbf{L}\mathbf{L}^\top \quad \text{(Cholesky decomposition)}$$

Then $\text{Cov}(\mathbf{Z}_{\text{corr}}) = \mathbf{L}\mathbf{I}\mathbf{L}^\top = \boldsymbol{\Sigma}$ ✓

In [ ]:
# Verify Cholesky decomposition manually
L = np.linalg.cholesky(Sigma.values)
Sigma_reconstructed = L @ L.T
max_error = np.max(np.abs(Sigma_reconstructed - Sigma.values))
print(f'Cholesky reconstruction max error: {max_error:.2e}  (should be ~0)')

# Run GBM simulation
gbm = simulate_gbm_paths(
    weights        = msr.weights,
    mu_annual      = mu.values,
    Sigma_annual   = Sigma.values,
    n_sims         = 5_000,
    horizon_years  = 1.0,
    tickers        = TICKERS,
    seed           = 42
)
print(f'\nGBM simulation shape: {gbm.paths.shape}  (n_sims x n_days+1)')
print(f'Median terminal value: {np.median(gbm.final_values):.4f}')

In [ ]:
fig = plot_gbm_simulation(gbm, label='Max Sharpe Portfolio')
plt.show()

---
## 10. Risk Measures: VaR & CVaR <a id='10-risk'></a>

### Value at Risk (VaR)
$$\text{VaR}_\alpha = -q_\alpha(R)$$

The loss that is **not exceeded with probability** $(1-\alpha)$. For $\alpha=5\%$, VaR$_{5\%}$ is the worst loss in 95% of scenarios.

**Limitation:** VaR says nothing about the size of losses beyond the threshold.

### Conditional Value at Risk (CVaR / Expected Shortfall)
$$\text{CVaR}_\alpha = -\mathbb{E}[R \mid R \leq q_\alpha(R)]$$

The **expected loss in the worst $\alpha\%$ of scenarios**. CVaR is:
- Always $\geq$ VaR at the same level
- A **coherent risk measure** (sub-additive → rewards diversification)
- The preferred measure for regulatory capital (FRTB, Basel III)

In [ ]:
# Compute risk metrics for all three portfolios
for label, weights in [('Max Sharpe', msr.weights), ('Min Variance', gmv.weights), ('Equal Weight', ew.weights)]:
    g = simulate_gbm_paths(weights, mu.values, Sigma.values, n_sims=10_000, seed=0)
    print(f'{label:15s}  |  VaR(5%): {g.var()*100:5.1f}%  |  CVaR(5%): {g.cvar()*100:5.1f}%  |'
          f'  Median return: {(np.median(g.final_values)-1)*100:+5.1f}%')

In [ ]:
# Rolling Sharpe ratio over the historical period
fig = plot_rolling_sharpe(log_ret, msr.weights, window=63)
plt.show()

---
## 11. Final Dashboard & Summary <a id='11-dashboard'></a>

In [ ]:
# Comprehensive results summary
results = pd.DataFrame([
    {
        'Portfolio':          label,
        'Ann. Return (%)':    round(p.ret * 100, 2),
        'Ann. Volatility (%)':round(p.vol * 100, 2),
        'Sharpe Ratio':       round(p.sharpe, 4),
        'Max Weight (%)':     round(p.weights.max() * 100, 1),
        'Min Weight (%)':     round(p.weights.min() * 100, 1),
    }
    for label, p in [('Max Sharpe', msr), ('Min Variance', gmv), ('Equal Weight', ew)]
])

print('\n=== PORTFOLIO OPTIMIZATION RESULTS ===')
print(results.to_string(index=False))

print('\n=== MAX SHARPE PORTFOLIO WEIGHTS ===')
w_df = pd.DataFrame({'Ticker': msr.tickers, 'Weight (%)': (msr.weights * 100).round(2)})
print(w_df.sort_values('Weight (%)', ascending=False).to_string(index=False))

In [ ]:
print('\n=== KEY INSIGHTS ===')
print(f'1. The Max Sharpe portfolio achieves Sharpe={msr.sharpe:.3f} vs Equal-Weight Sharpe={ew.sharpe:.3f}')
print(f'   Improvement: {(msr.sharpe/ew.sharpe - 1)*100:.1f}% better risk-adjusted return')

avg_vol = np.sqrt(np.diag(Sigma.values)).mean()
print(f'\n2. Average single-asset volatility: {avg_vol*100:.1f}%')
print(f'   Max Sharpe portfolio volatility: {msr.vol*100:.1f}%  ({(avg_vol - msr.vol)*100:.1f}pp diversification benefit)')

print(f'\n3. GBM 1-year VaR(5%):  {gbm.var()*100:.1f}%')
print(f'   GBM 1-year CVaR(5%): {gbm.cvar()*100:.1f}%')
print(f'   Interpretation: In the worst 5% of scenarios, we expect to lose at least {gbm.cvar()*100:.1f}% of portfolio value over 1 year.')